# Lab · Huấn luyện theo kiểu PyTorch · LỜI GIẢI

> Chỉ mở file này khi bạn đã thử bài tập, hoặc đã dùng hết cả 3 gợi ý.

Bài tập cho [`lesson.md`](../lesson.md) (Module 4). Gom bốn lab của khóa theo thứ tự: **A** SGD và `DataLoader`, **B** mini-batch, **C** `optim.SGD`, **D** tách train/validation và chọn learning rate.

**Cách làm:** tự thử trước. Kẹt thì mở **Gợi ý 1**, rồi **Gợi ý 2**, cuối cùng mới **Gợi ý 3**. Lời giải ở thư mục `solution/`.

Môi trường do **uv** quản lý: trong thư mục khóa `04-intro-neural-networks-pytorch/`, chạy `uv sync` (một lần), rồi `uv run jupyter lab`.

## Chuẩn bị

In [ ]:
import os, tempfile
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
%matplotlib inline

print("PyTorch", torch.__version__)

## Bài 1. A · Dataset và `DataLoader` với batch size 1

Viết lớp `Data(Dataset)`: `self.x = torch.arange(-3, 3, 0.1).view(-1, 1)`, `self.f = 1 * self.x - 1`, `self.y = self.f + 0.1 * torch.randn(self.x.size())`, `self.len = self.x.shape[0]`; `__getitem__` trả `(x[i], y[i])`, `__len__` trả `self.len`.

Đặt `torch.manual_seed(1)` rồi tạo `dataset = Data()`. Tạo `loader1 = DataLoader(dataset, batch_size=1)`.
Lưu `iters1 = len(loader1)` và `first_batch = next(iter(loader1))`.

In [ ]:
class Data(Dataset):
    def __init__(self):
        self.x = torch.arange(-3, 3, 0.1).view(-1, 1)
        self.f = 1 * self.x - 1
        self.y = self.f + 0.1 * torch.randn(self.x.size())
        self.len = self.x.shape[0]

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.len

torch.manual_seed(1)
dataset = Data()
loader1 = DataLoader(dataset=dataset, batch_size=1)
iters1 = len(loader1)
first_batch = next(iter(loader1))
print(iters1, first_batch)

In [ ]:
assert len(dataset) == 60
assert iters1 == 60, "batch size 1 → 60 iteration mỗi epoch"
x0, y0 = first_batch
assert tuple(x0.shape) == (1, 1) and tuple(y0.shape) == (1, 1), "mỗi lô có thêm chiều batch"
assert abs(x0.item() + 3.0) < 1e-5
print("✅ Bài 1 đúng")

## Bài 2. A · Huấn luyện SGD bằng tay với `DataLoader`

Viết `criterion(yhat, y)` = MSE, và hàm `train_manual(batch_size, epochs=10, lr=0.1)`:
- tạo `loader = DataLoader(dataset, batch_size=batch_size)`;
- `w = -15.0`, `b = -10.0` (có gradient);
- mỗi **epoch**: trước hết ghi cost trên **toàn bộ** dữ liệu `criterion(w * dataset.x + b, dataset.y)` vào list `LOSS`; rồi duyệt từng lô: dự đoán, loss, `backward()`, cập nhật dưới `torch.no_grad()`, đặt gradient về 0;
- trả về `(w.item(), b.item(), LOSS)`.

Chạy với `batch_size=1` (đó là SGD): lưu `w_sgd, b_sgd, LOSS_sgd`.

In [ ]:
def criterion(yhat, y):
    return torch.mean((yhat - y) ** 2)

def train_manual(batch_size, epochs=10, lr=0.1):
    loader = DataLoader(dataset, batch_size=batch_size)
    w = torch.tensor(-15.0, requires_grad=True)
    b = torch.tensor(-10.0, requires_grad=True)
    LOSS = []
    for epoch in range(epochs):
        LOSS.append(criterion(w * dataset.x + b, dataset.y).item())
        for x, y in loader:
            loss = criterion(w * x + b, y)
            loss.backward()
            with torch.no_grad():
                w -= lr * w.grad
                b -= lr * b.grad
            w.grad.zero_()
            b.grad.zero_()
    return w.item(), b.item(), LOSS

w_sgd, b_sgd, LOSS_sgd = train_manual(batch_size=1)
print(w_sgd, b_sgd, LOSS_sgd)

In [ ]:
assert len(LOSS_sgd) == 10
assert LOSS_sgd[0] > 800 and LOSS_sgd[-1] < 0.1
assert abs(w_sgd - 1.0) < 0.2 and abs(b_sgd + 1.0) < 0.2
print("✅ Bài 2 đúng: SGD hội tụ chỉ sau vài epoch")

## Bài 3. A · So với batch gradient descent

Huấn luyện với `batch_size = len(dataset)` (mỗi lần dùng **cả tập**: đó là batch GD), cùng 10 epoch, `lr = 0.1`. Lưu `LOSS_bgd`.
Vẽ `LOSS_bgd` và `LOSS_sgd`. Lưu `bgd_slower = LOSS_bgd[-1] > LOSS_sgd[-1]`.
Câu hỏi: vì sao SGD tiến nhanh hơn tính theo epoch? (Gợi ý: đếm số lần cập nhật.)

In [ ]:
_, _, LOSS_bgd = train_manual(batch_size=len(dataset))
bgd_slower = LOSS_bgd[-1] > LOSS_sgd[-1]
plt.plot(LOSS_bgd, label="Batch GD")
plt.plot(LOSS_sgd, label="SGD")
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("cost"); plt.legend(); plt.show()
print(bgd_slower, LOSS_bgd[-1], LOSS_sgd[-1])

In [ ]:
assert bgd_slower is True
print("✅ Bài 3 đúng: SGD cập nhật 600 lần, batch GD chỉ 10 lần trong cùng 10 epoch")

## Bài 4. B · Đếm iteration và lô cuối

Lập dictionary `iters` cho các batch size `1, 5, 10, 20`: giá trị là số lô (iteration mỗi epoch) của `DataLoader(dataset, batch_size=bs)`.
Rồi với `batch_size=7`:
- `n7`: số lô;
- `n7_drop`: số lô khi dùng `drop_last=True`;
- `last_len`: số mẫu trong lô **cuối** (khi không `drop_last`).

In [ ]:
iters = {bs: len(DataLoader(dataset, batch_size=bs)) for bs in (1, 5, 10, 20)}
n7 = len(DataLoader(dataset, batch_size=7))
n7_drop = len(DataLoader(dataset, batch_size=7, drop_last=True))
for x, y in DataLoader(dataset, batch_size=7):
    last_len = len(x)
print(iters, n7, n7_drop, last_len)

In [ ]:
assert iters == {1: 60, 5: 12, 10: 6, 20: 3}
assert n7 == 9 and n7_drop == 8, "60 = 8×7 + 4 nên có 9 lô, lô cuối 4 mẫu"
assert last_len == 4
print("✅ Bài 4 đúng")

## Bài 5. B · Mini-batch với batch size 5, 10, 20

Huấn luyện mini-batch với `batch_size` = 5, 10, 20 (10 epoch, `lr = 0.1`). Lưu `final_loss = {5: ..., 10: ..., 20: ...}` (cost cuối của từng cách) và `updates = {bs: số lần cập nhật}` (iteration mỗi epoch × 10).
Vẽ cả năm đường loss: batch GD, SGD, mini-batch 5, 10, 20.

In [ ]:
final_loss, updates = {}, {}
curves = {"Batch GD": LOSS_bgd, "SGD": LOSS_sgd}
for bs in (5, 10, 20):
    _, _, L = train_manual(bs)
    final_loss[bs] = L[-1]
    updates[bs] = len(DataLoader(dataset, batch_size=bs)) * 10
    curves[f"mini-batch {bs}"] = L
for name, L in curves.items():
    plt.plot(L, label=name)
plt.yscale("log"); plt.xlabel("epoch"); plt.ylabel("cost"); plt.legend(); plt.show()
print(final_loss, updates)

In [ ]:
assert set(final_loss) == {5, 10, 20}
assert all(v < 0.05 for v in final_loss.values()), "cả ba đều hội tụ tốt trong 10 epoch"
assert updates == {5: 120, 10: 60, 20: 30}
print("✅ Bài 5 đúng")

## Bài 6. C · Mô hình, hàm loss và optimizer

Viết lớp `linear_regression(nn.Module)` (như Module 3), rồi tạo:
- `model = linear_regression(1, 1)`;
- `loss_fn = nn.MSELoss()`;
- `optimizer = optim.SGD(model.parameters(), lr=0.01)`.

Đặt tham số khởi đầu `weight = -15`, `bias = -10` bằng `model.state_dict()['linear.weight'][0] = ...` (và tương tự cho bias).
Lưu `lr_in_opt` là learning rate đọc từ `optimizer.state_dict()['param_groups'][0]['lr']`.

In [ ]:
class linear_regression(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()
        self.linear = nn.Linear(input_size, output_size)

    def forward(self, x):
        return self.linear(x)

model = linear_regression(1, 1)
loss_fn = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)
model.state_dict()['linear.weight'][0] = -15
model.state_dict()['linear.bias'][0] = -10
lr_in_opt = optimizer.state_dict()['param_groups'][0]['lr']
print(list(model.parameters()), lr_in_opt)

In [ ]:
assert lr_in_opt == 0.01
assert model.linear.weight.item() == -15 and model.linear.bias.item() == -10
print("✅ Bài 6 đúng")

## Bài 7. C · Vòng lặp huấn luyện với optimizer

Huấn luyện `model` ở bài 6 trên `DataLoader(dataset, batch_size=1)`, 10 epoch, dùng đúng năm bước: forward, loss, `optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`.
Sau đó lưu `w_opt = model.linear.weight.item()` và `b_opt = model.linear.bias.item()`.

In [ ]:
for epoch in range(10):
    for x, y in DataLoader(dataset, batch_size=1):
        yhat = model(x)
        loss = loss_fn(yhat, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
w_opt = model.linear.weight.item()
b_opt = model.linear.bias.item()
print(w_opt, b_opt)

In [ ]:
assert abs(w_opt - 1.0) < 0.15 and abs(b_opt + 1.0) < 0.15, "tham số phải gần w = 1, b = -1"
print("✅ Bài 7 đúng: cùng kết quả với vòng lặp viết tay, nhưng gọn hơn")

## Bài 8. C · Luyện tập: batch GD bằng optimizer

Tạo mô hình **mới** `model_b` (đặt weight −15, bias −10 như trên) và `optimizer_b = optim.SGD(model_b.parameters(), lr=0.1)`.
Huấn luyện **batch GD** (lô là cả tập) 10 epoch. Lưu `cost_b`: MSE của `model_b` trên toàn bộ `dataset.x`, `dataset.y` (không cần gradient).

In [ ]:
model_b = linear_regression(1, 1)
model_b.state_dict()['linear.weight'][0] = -15
model_b.state_dict()['linear.bias'][0] = -10
optimizer_b = optim.SGD(model_b.parameters(), lr=0.1)
for epoch in range(10):
    for x, y in DataLoader(dataset, batch_size=len(dataset)):
        loss = loss_fn(model_b(x), y)
        optimizer_b.zero_grad()
        loss.backward()
        optimizer_b.step()
with torch.no_grad():
    cost_b = loss_fn(model_b(dataset.x), dataset.y).item()
print(cost_b)

In [ ]:
assert cost_b < 2.0, "sau 10 lần cập nhật cost đã giảm từ ~835 xuống còn vài đơn vị"
assert cost_b > 0.05, "batch GD 10 epoch vẫn chưa hội tụ hẳn như SGD"
print("✅ Bài 8 đúng")

## Bài 9. D · Dữ liệu train có ngoại lai, validation sạch

Viết lớp `DataSplit(Dataset)` với tham số `train=True`:
`x = torch.arange(-3, 3, 0.1).view(-1, 1)`, `f = -3 * x + 1`, `y = f + 0.1 * torch.randn(x.size())`.
Nếu `train`: tạo **ngoại lai** bằng `y[0] = 0` và `y[50:55] = 20`; nếu không thì để nguyên.
Đặt `torch.manual_seed(1)`, tạo `train_data = DataSplit()` và `val_data = DataSplit(train=False)`.

In [ ]:
class DataSplit(Dataset):
    def __init__(self, train=True):
        self.x = torch.arange(-3, 3, 0.1).view(-1, 1)
        self.f = -3 * self.x + 1
        self.y = self.f + 0.1 * torch.randn(self.x.size())
        self.len = self.x.shape[0]
        if train:
            self.y[0] = 0
            self.y[50:55] = 20

    def __getitem__(self, index):
        return self.x[index], self.y[index]

    def __len__(self):
        return self.len

torch.manual_seed(1)
train_data = DataSplit()
val_data = DataSplit(train=False)
plt.plot(train_data.x.numpy(), train_data.y.numpy(), "xr", label="train (có ngoại lai)")
plt.plot(val_data.x.numpy(), val_data.y.numpy(), ".b", label="validation")
plt.plot(train_data.x.numpy(), train_data.f.numpy(), label="hàm thật")
plt.legend(); plt.show()

In [ ]:
assert (train_data.y[50:55] == 20).all() and train_data.y[0].item() == 0
assert (val_data.y[50:55] < 0).all(), "validation không có ngoại lai"
print("✅ Bài 9 đúng")

## Bài 10. D · Chọn learning rate bằng validation

Với `learning_rates = [0.0001, 0.001, 0.01, 0.1]`: với mỗi giá trị, **đặt `torch.manual_seed(1)`**, tạo mô hình `linear_regression(1, 1)`, `optim.SGD(..., lr=lr)`, huấn luyện 10 epoch trên `DataLoader(train_data, batch_size=1)` (loss MSE, năm bước chuẩn).
Sau đó ghi **training error** (MSE trên toàn `train_data`) và **validation error** (MSE trên toàn `val_data`, dưới `torch.no_grad()`) vào hai tensor `train_error`, `validation_error` (độ dài 4), và lưu mô hình vào list `MODELS`.
Rồi:
- `best_val_idx`: chỉ số có validation error nhỏ nhất;
- `best_train_idx`: chỉ số có training error nhỏ nhất.
**Đoán trước:** hai chỉ số này có trùng nhau không?

In [ ]:
learning_rates = [0.0001, 0.001, 0.01, 0.1]
train_error = torch.zeros(len(learning_rates))
validation_error = torch.zeros(len(learning_rates))
MODELS = []
for i, lr in enumerate(learning_rates):
    torch.manual_seed(1)
    m = linear_regression(1, 1)
    opt = optim.SGD(m.parameters(), lr=lr)
    for epoch in range(10):
        for x, y in DataLoader(train_data, batch_size=1):
            loss = loss_fn(m(x), y)
            opt.zero_grad()
            loss.backward()
            opt.step()
    with torch.no_grad():
        train_error[i] = loss_fn(m(train_data.x), train_data.y).item()
        validation_error[i] = loss_fn(m(val_data.x), val_data.y).item()
    MODELS.append(m)
best_val_idx = int(validation_error.argmin())
best_train_idx = int(train_error.argmin())
plt.semilogx(learning_rates, train_error.numpy(), label="training error")
plt.semilogx(learning_rates, validation_error.numpy(), label="validation error")
plt.xlabel("learning rate"); plt.ylabel("MSE"); plt.legend(); plt.show()
print(train_error, validation_error, best_train_idx, best_val_idx)

In [ ]:
assert len(MODELS) == 4
assert best_val_idx == int(validation_error.argmin())
assert validation_error[0] > validation_error[best_val_idx], "lr = 0.0001 quá nhỏ: học chưa kịp"
print("best theo train:", learning_rates[best_train_idx], "| best theo validation:", learning_rates[best_val_idx])
print("✅ Bài 10 đúng. Chọn mô hình theo VALIDATION, không theo training.")

## Bài 11. D · Lưu và nạp mô hình tốt nhất

Lấy `best_model = MODELS[best_val_idx]`.
- Lưu `state_dict` vào file tạm: `path = os.path.join(tempfile.gettempdir(), "best_model.pt")`, bằng `torch.save`.
- Tạo `loaded = linear_regression(1, 1)` (tham số khác), nạp tham số từ file bằng `load_state_dict`, rồi `.eval()`.
- `same = True` nếu `loaded` và `best_model` cho **cùng** dự đoán trên `val_data.x`.

In [ ]:
best_model = MODELS[best_val_idx]
path = os.path.join(tempfile.gettempdir(), "best_model.pt")
torch.save(best_model.state_dict(), path)

torch.manual_seed(123)
loaded = linear_regression(1, 1)
loaded.load_state_dict(torch.load(path))
loaded.eval()
with torch.no_grad():
    same = torch.allclose(loaded(val_data.x), best_model(val_data.x))
print(same)

In [ ]:
assert os.path.exists(path)
assert same is True
print("✅ Bài 11 đúng: không cần huấn luyện lại để dùng mô hình")

## Bài 12. D · Đánh giá cuối: không cập nhật, không theo dõi gradient

Tính `test_loss`: MSE của `best_model` trên `val_data` (coi đây là bước đánh giá) **bên trong** `torch.no_grad()`.
Rồi lưu `no_graph = not test_loss.requires_grad`.
Câu hỏi: nếu không bọc `no_grad`, kết quả có gì khác? Vì sao khi đánh giá ta không gọi `backward()` và `step()`?

In [ ]:
with torch.no_grad():
    test_loss = loss_fn(best_model(val_data.x), val_data.y)
no_graph = not test_loss.requires_grad
print(test_loss, no_graph)

In [ ]:
assert no_graph is True
assert test_loss.item() == validation_error[best_val_idx].item()
print("✅ Bài 12 đúng")

## Hết